# PRAGma App — LOT 기반 에칭 공정 분석 시스템

**파이프라인:**
1. LOT 번호 입력 → MES(CSV Mock)에서 공정 데이터 자동 조회
2. LightGBM 모델로 에칭 속도 예측
3. **EXAONE Model RAG** (LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct + Hybrid BM25+Vector + ML 예측/OPLS 컨텍스트 주입)으로 공정 해석 및 조치 방안 제시

**Google Colab 실행 필수** — EXAONE은 GPU(4bit 양자화)가 필요합니다.  
**Secrets에 `HF_TOKEN` 필요** — HuggingFace 인증용

## 0. 환경 설정

In [2]:
# 1. Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 2. GitHub에서 코드 가져오기
import subprocess
result = subprocess.run(["git", "clone", "https://github.com/heyitsmialee/PRAGma.git"],
                        capture_output=True, text=True)
if "already exists" in result.stderr:
    print("PRAGma 이미 존재 — pull로 최신화")
    subprocess.run(["git", "-C", "/content/PRAGma", "pull"], check=True)
else:
    print(result.stdout or result.stderr)

import os
os.chdir("/content/PRAGma")
print(f"작업 디렉토리: {os.getcwd()}")

Mounted at /content/drive
Cloning into 'PRAGma'...

작업 디렉토리: /content/PRAGma


In [3]:
import os
import json
import pickle
import subprocess
import sys
import numpy as np
import pandas as pd
from pathlib import Path

# ── 패키지 설치 ───────────────────────────────────────────────────────────────
PACKAGES = [
    "langchain", "langchain-core", "langchain-community",
    "langchain-text-splitters", "langchain-huggingface",
    "langchain-chroma", "rank_bm25",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U"] + PACKAGES, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.52.0", "accelerate", "bitsandbytes", "torch"], check=True)
print("패키지 설치 완료")

# ── HuggingFace 토큰 (Colab Secrets) ─────────────────────────────────────────
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Secrets에 HF_TOKEN이 없습니다. 왼쪽 🔑 → Add new secret 으로 추가하세요.")

from huggingface_hub import login
login(token=HF_TOKEN)
print("HuggingFace 로그인 완료")

# ── 경로 설정 ─────────────────────────────────────────────────────────────────
# GitHub clone 경로 우선, Drive fallback
GITHUB_DIR = Path("/content/PRAGma")
DRIVE_DIR  = Path("/content/drive/MyDrive/Colab Notebooks/PRAGma")

MODEL_PATH      = GITHUB_DIR / "notebooks/best_LightGBM_mass_speed_regressor.pkl"
CSV_PATH        = GITHUB_DIR / "data/Train_0319.csv"
PAPER_JSON_PATH = GITHUB_DIR / "data/rag_data_all.json"
SHAP_MD_PATH    = GITHUB_DIR / "notebooks/shap_analysis_for_rag.md"
CHROMA_DIR      = "/content/pragma_chroma"

# Drive fallback (GitHub에 없는 경우)
if not MODEL_PATH.exists():
    MODEL_PATH = DRIVE_DIR / "best_LightGBM_mass_speed_regressor.pkl"
if not CSV_PATH.exists():
    CSV_PATH   = DRIVE_DIR / "Train_0319.csv"

print(f"모델 경로  : {MODEL_PATH}")
print(f"CSV 경로   : {CSV_PATH}")
print(f"JSON 경로  : {PAPER_JSON_PATH}")
print(f"SHAP MD    : {SHAP_MD_PATH}")
print("설정 완료")

패키지 설치 완료
HuggingFace 로그인 완료
모델 경로  : /content/drive/MyDrive/Colab Notebooks/PRAGma/best_LightGBM_mass_speed_regressor.pkl
CSV 경로   : /content/PRAGma/data/Train_0319.csv
JSON 경로  : /content/PRAGma/data/rag_data_all.json
SHAP MD    : /content/PRAGma/notebooks/shap_analysis_for_rag.md
설정 완료


## 1. 모델 및 데이터 로드

In [4]:
# LightGBM 모델 로드
with open(MODEL_PATH, "rb") as f:
    lgbm_model = pickle.load(f)

MODEL_FEATURES = lgbm_model.feature_name_
print(f"모델 로드 완료 — 피처 수: {len(MODEL_FEATURES)}")
print(f"피처 목록: {MODEL_FEATURES[:8]} ...")

# MES Mock 데이터 로드 (실제 환경에서는 MES API 호출로 대체)
df_mes = pd.read_csv(CSV_PATH, encoding="cp949")
print(f"\nMES Mock 데이터 로드 완료 — LOT 수: {len(df_mes)}, 컬럼 수: {len(df_mes.columns)}")
print(f"LOT 예시: {df_mes['LOT'].head(5).tolist()}")

# RAG 지식베이스 로드
with open(PAPER_JSON_PATH, "r", encoding="utf-8") as f:
    paper_rules = json.load(f)

shap_content = SHAP_MD_PATH.read_text(encoding="utf-8") if SHAP_MD_PATH.exists() else ""

print(f"\nRAG 지식베이스 로드 완료")
print(f"  논문 rule 수: {len(paper_rules)}")
print(f"  SHAP 분석 MD 존재: {bool(shap_content)}")

모델 로드 완료 — 피처 수: 45
피처 목록: ['cu_thick_max', 'cu_thick_avg', 'cu_thick_min', 'cu_thick_std', 'cu_thick_median', 'etch_factor', 'meas_etch_cu', 'meas_etch_hcl'] ...

MES Mock 데이터 로드 완료 — LOT 수: 8166, 컬럼 수: 103
LOT 예시: ['A20000', 'A20001', 'A20002', 'A20003', 'A20004']

RAG 지식베이스 로드 완료
  논문 rule 수: 5
  SHAP 분석 MD 존재: False


## 2. CSV 컬럼 → 모델 피처 매핑

실제 MES에서 받는 raw 컬럼명을 LightGBM 모델 입력 피처명으로 변환합니다.

In [5]:
# ── 연속형 컬럼 매핑 (동적 탐색) ─────────────────────────────────────────────
CONTINUOUS_MAP = {
    "Cu 표면두께 Max_Val"   : "cu_thick_max",
    "Cu 표면두께 AVG_VAL"   : "cu_thick_avg",
    "Cu 표면두께 Min_Val"   : "cu_thick_min",
    "Cu 표면두께 Std_Val"   : "cu_thick_std",
    "Cu 표면두께 Median_Val": "cu_thick_median",
}

_feat_suffix_map = {
    "Etch factor"             : "etch_factor",
    "Etching(염화동) - Cu"    : "meas_etch_cu",
    "Etching(염화동) - HCl"   : "meas_etch_hcl",
    "Etching(염화동) - 비중"  : "meas_etch_sg",
    "Etching(염화동) - 온도"  : "meas_etch_temp",
    "Etching-첨가제(HB-120EF)": "meas_etch_additive",
    "Etching량"               : "meas_etch_amount",
    "Soft Etch - Cu"          : "meas_softetch_cu",
    "Soft Etch - H2SO4"       : "meas_softetch_h2so4",
    "Soft Etch - SPS"         : "meas_softetch_sps",
    "박리액 - 농도"           : "meas_strip_conc",
    "수세수 - pH"             : "meas_rinse_ph",
    "현상액 - pH"             : "meas_dev_ph",
    "현상액 - 농도"           : "meas_dev_conc",
}

for col in df_mes.columns:
    if "분석치" in col:
        suffix = col.split("_", 1)[-1] if "_" in col else col
        if suffix in _feat_suffix_map:
            CONTINUOUS_MAP[col] = _feat_suffix_map[suffix]

CONTINUOUS_MAP_RESOLVED = CONTINUOUS_MAP

print(f"매핑 완료: {len(CONTINUOUS_MAP_RESOLVED)}개 연속형 피처")
for k, v in CONTINUOUS_MAP_RESOLVED.items():
    print(f"  {k!r:45s} → {v}")

매핑 완료: 19개 연속형 피처
  'Cu 표면두께 Max_Val'                             → cu_thick_max
  'Cu 표면두께 AVG_VAL'                             → cu_thick_avg
  'Cu 표면두께 Min_Val'                             → cu_thick_min
  'Cu 표면두께 Std_Val'                             → cu_thick_std
  'Cu 표면두께 Median_Val'                          → cu_thick_median
  '분석치_Etch factor'                             → etch_factor
  '분석치_Etching(염화동) - Cu'                       → meas_etch_cu
  '분석치_Etching(염화동) - HCl'                      → meas_etch_hcl
  '분석치_Etching(염화동) - 비중'                       → meas_etch_sg
  '분석치_Etching(염화동) - 온도'                       → meas_etch_temp
  '분석치_Etching-첨가제(HB-120EF)'                   → meas_etch_additive
  '분석치_Etching량'                                → meas_etch_amount
  '분석치_Soft Etch - Cu'                          → meas_softetch_cu
  '분석치_Soft Etch - H2SO4'                       → meas_softetch_h2so4
  '분석치_Soft Etch - SPS'                         → meas_softetch_sps
  '분석치

[링크 텍스트](https://)## 3. MES Mock 함수

실제 환경에서는 MES REST API 호출로 교체합니다.

In [6]:
def get_lot_data(lotno: str) -> dict | None:
    """MES Mock: LOT 번호 → 공정 데이터 딕셔너리 반환

    실제 MES 연동 시 이 함수만 교체하면 됩니다:
        response = requests.get(f"{MES_URL}/lot/{lotno}")
        return response.json()
    """
    rows = df_mes[df_mes["LOT"] == lotno]
    if rows.empty:
        print(f"[경고] LOT '{lotno}' 를 찾을 수 없습니다.")
        print(f"  사용 가능한 LOT 예시: {df_mes['LOT'].head(10).tolist()}")
        return None
    return rows.iloc[0].to_dict()


# 테스트
sample = get_lot_data("A20000")
if sample:
    print(f"LOT A20000 조회 성공")
    print(f"  실제 에칭 속도: {sample.get('부식 Speed')} m/min")
    print(f"  에칭 온도: {sample.get('분析치_Etching(염화동) - 온도')} °C")
    print(f"  에칭 비중: {sample.get('분析치_Etching(염화동) - 비중')}")

LOT A20000 조회 성공
  실제 에칭 속도: 2.8 m/min
  에칭 온도: None °C
  에칭 비중: None


## 4. 피처 변환 (MES 데이터 → LightGBM 입력)

In [7]:
# ── 범주형 컬럼 정의 ───────────────────────────────────────────────────────────
CATEGORICAL_COLS = {
    "재작업사유" : {
        "prefix": "rework_history",
        "values": ["Unknown", "기타", "기판 겹침", "두께 미달", "딤플", "설비 에러"],
        "sep"   : "_",   # 모델은 공백을 언더스코어로 저장했음
    },
    "노광 설비정보": {
        "prefix": "expo_eq_id",
        "values": [f"EXP-{i:03d}" for i in range(1, 8)],
        "sep"   : "_",
    },
    "DES 설비정보": {
        "prefix": "des_eq_id",
        "values": [f"DES-{i:03d}" for i in range(1, 7)],
        "sep"   : "_",
    },
    "정면 설비정보": {
        "prefix": "brush_eq_id",
        "values": [f"PRE-{i:03d}" for i in range(1, 8)],
        "sep"   : "_",
    },
}


def prepare_features(lot_data: dict) -> pd.DataFrame:
    """MES 딕셔너리 → LightGBM 입력 DataFrame (1행)"""
    row = {}

    # 1. 연속형 피처
    for csv_col, feat_name in CONTINUOUS_MAP_RESOLVED.items():
        val = lot_data.get(csv_col, np.nan)
        row[feat_name] = float(val) if val is not None and str(val) not in ("", "nan", "None") else np.nan

    # 2. 범주형 → one-hot
    for csv_col, cfg in CATEGORICAL_COLS.items():
        raw_val = str(lot_data.get(csv_col, "")).strip()
        # 공백을 언더스코어로 변환 (rework_history 한정)
        norm_val = raw_val.replace(" ", "_") if cfg["prefix"] == "rework_history" else raw_val
        for v in cfg["values"]:
            norm_v = v.replace(" ", "_") if cfg["prefix"] == "rework_history" else v
            feat_key = f"{cfg['prefix']}{cfg['sep']}{norm_v}"
            row[feat_key] = 1 if norm_val == norm_v else 0

    # 3. 모델이 요구하는 피처 순서로 정렬, 없는 피처는 0 채움
    for feat in MODEL_FEATURES:
        if feat not in row:
            row[feat] = 0

    return pd.DataFrame([row])[MODEL_FEATURES]


# 테스트
if sample:
    X = prepare_features(sample)
    print(f"피처 변환 완료: shape = {X.shape}")
    print(X[[
        "cu_thick_avg", "etch_factor", "meas_etch_temp",
        "meas_etch_sg", "meas_etch_cu", "expo_eq_id_EXP-001"
    ]].to_string(index=False))

피처 변환 완료: shape = (1, 45)
 cu_thick_avg  etch_factor  meas_etch_temp  meas_etch_sg  meas_etch_cu  expo_eq_id_EXP-001
    17.919439     5.291029       48.121027      1.369145    150.468409                   1


## 5. ML 예측 (LightGBM)

> 인용구 추가



In [8]:
def predict_etch_speed(lot_data: dict) -> tuple[float, pd.DataFrame]:
    """LightGBM으로 에칭 속도 예측

    Returns:
        (예측값, 피처 DataFrame)
    """
    X = prepare_features(lot_data)
    pred = lgbm_model.predict(X)[0]
    return float(pred), X


def get_opls_bounds() -> dict:
    """에칭 공정 OPLS 기준값 (하드코딩 — 실제는 MES에서 조회)"""
    return {
        "meas_etch_temp"    : {"lcl": 44.5, "sl": 48.0, "ucl": 53.0, "unit": "°C",  "name": "에칭 온도"},
        "meas_etch_sg"      : {"lcl": 1.32,  "sl": 1.37,  "ucl": 1.42,  "unit": "",    "name": "에칭 비중"},
        "meas_etch_cu"      : {"lcl": 125.0, "sl": 155.0, "ucl": 185.0, "unit": "g/L", "name": "에칭 Cu 농도"},
        "meas_etch_hcl"     : {"lcl": 0.3,   "sl": 0.5,   "ucl": 0.7,   "unit": "N",   "name": "에칭 HCl"},
        "meas_etch_additive": {"lcl": 2.6,   "sl": 3.0,   "ucl": 3.4,   "unit": "g/L", "name": "에칭 첨가제"},
    }


def check_opls_status(X: pd.DataFrame) -> list[dict]:
    """OPLS 기준 대비 이탈 항목 체크"""
    bounds = get_opls_bounds()
    alerts = []
    for feat, lim in bounds.items():
        if feat not in X.columns:
            continue
        val = X[feat].iloc[0]
        if pd.isna(val):
            continue
        status = "정상"
        if val > lim["ucl"]:
            status = "UCL 초과 (상한 이탈)"
        elif val < lim["lcl"]:
            status = "LCL 미달 (하한 이탈)"
        elif val > lim["sl"] * 1.02:
            status = "SL 상향 근접"
        elif val < lim["sl"] * 0.98:
            status = "SL 하향 근접"
        alerts.append({
            "피처"  : feat,
            "항목"  : lim["name"],
            "현재값": round(val, 4),
            "LCL"   : lim["lcl"],
            "SL"    : lim["sl"],
            "UCL"   : lim["ucl"],
            "단위"  : lim["unit"],
            "상태"  : status,
        })
    return alerts


# 테스트
if sample:
    pred_speed, X = predict_etch_speed(sample)
    actual_speed  = sample.get("부식 Speed", "N/A")
    print(f"=== LOT: {sample['LOT']} ===")
    print(f"예측 에칭 속도 : {pred_speed:.4f} m/min")
    print(f"실제 에칭 속도 : {actual_speed} m/min")
    if isinstance(actual_speed, (int, float)):
        err = abs(pred_speed - actual_speed) / actual_speed * 100
        print(f"오차율          : {err:.2f}%")

    print("\n=== OPLS 상태 ===" )
    alerts = check_opls_status(X)
    df_alerts = pd.DataFrame(alerts)
    print(df_alerts.to_string(index=False))

=== LOT: A20000 ===
예측 에칭 속도 : 1.9959 m/min
실제 에칭 속도 : 2.8 m/min
오차율          : 28.72%

=== OPLS 상태 ===
                피처       항목      현재값    LCL     SL    UCL  단위       상태
    meas_etch_temp    에칭 온도  48.1210  44.50  48.00  53.00  °C       정상
      meas_etch_sg    에칭 비중   1.3691   1.32   1.37   1.42           정상
      meas_etch_cu 에칭 Cu 농도 150.4684 125.00 155.00 185.00 g/L SL 하향 근접
     meas_etch_hcl   에칭 HCl   0.5198   0.30   0.50   0.70   N SL 상향 근접
meas_etch_additive   에칭 첨가제   3.0212   2.60   3.00   3.40 g/L       정상


## 6. RAG 지식베이스 준비

In [9]:
def build_knowledge_context(max_rules: int = 5) -> str:
    """RAG 지식베이스를 LLM 시스템 프롬프트용 텍스트로 변환"""
    parts = []

    # 논문 기반 공정 rule (상위 N개)
    if paper_rules:
        parts.append("[논문 기반 공정 규칙]")
        for item in paper_rules[:max_rules]:
            parts.append(json.dumps(item, ensure_ascii=False, indent=2))

    # SHAP 분석 결과
    if shap_content:
        parts.append("\n[SHAP 모델 해석 분석]")
        parts.append(shap_content[:3000])  # 토큰 절약

    return "\n\n".join(parts)


def retrieve_relevant_rules(question: str, lot_data: dict, top_k: int = 3) -> str:
    """키워드 기반 관련 rule 검색 (경량 RAG)"""
    keywords = []
    q_lower  = question.lower()

    # 질문 키워드 추출
    keyword_map = {
        "온도"   : ["온도", "temperature", "temp"],
        "비중"   : ["비중", "density", "sg"],
        "속도"   : ["속도", "speed", "컨베이어"],
        "Cu"     : ["cu", "구리", "copper"],
        "불량"   : ["불량", "defect", "이상", "문제"],
        "수율"   : ["수율", "yield"],
        "SHAP"   : ["shap", "중요도", "영향"],
        "과에칭" : ["과에칭", "over-etch", "선폭"],
        "잔동"   : ["잔동", "under-etch"],
    }
    for key, terms in keyword_map.items():
        if any(t in q_lower for t in terms):
            keywords.append(key)

    # paper_rules에서 키워드 매칭
    scored = []
    for item in paper_rules:
        text  = json.dumps(item, ensure_ascii=False).lower()
        score = sum(1 for kw in keywords if kw.lower() in text)
        if score > 0:
            scored.append((score, item))

    scored.sort(key=lambda x: -x[0])
    top_rules = [json.dumps(item, ensure_ascii=False, indent=2) for _, item in scored[:top_k]]

    if not top_rules and paper_rules:
        top_rules = [json.dumps(paper_rules[0], ensure_ascii=False, indent=2)]

    result = "\n---\n".join(top_rules)
    if shap_content:
        result += "\n\n[SHAP 분석]\n" + shap_content[:2000]
    return result


# 테스트
ctx = retrieve_relevant_rules("에칭 온도가 높을 때 문제가 뭐가 있나요?", {})
print(f"검색된 컨텍스트 길이: {len(ctx)} 자")
print(ctx[:500])

검색된 컨텍스트 길이: 4979 자
{
  "paper_id": "air_regenerated_cupric_chloride",
  "citation": {
    "title": "Copper Etching in Air Regenerated Cupric Chloride Solution",
    "year": 2010,
    "source": "Cupric chloride process study"
  },
  "korean_summary": "이 논문은 air-regenerated cupric chloride 용액에서 temperature, specific gravity, free acid concentration이 평균 식각속도에 미치는 영향을 다룬다. 네 변수 기준으로 meas_etch_temp, meas_etch_sg, meas_etch_hcl과 직접 또는 준직접 매핑이 가능하다.",
  "mapped_variables": [
    {
      "paper_term": "temperature",
     


In [10]:
## 7-a. EXAONE LLM + Hybrid RAG 초기화 (최초 1회 — GPU 필요)
import torch
from transformers import BitsAndBytesConfig
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List as _List

E5_PREFIX = "Instruct: 공정 이상 원인과 조치 방법을 찾으세요\nQuery: "

if not torch.cuda.is_available():
    raise RuntimeError("GPU가 없습니다. Colab Runtime → 런타임 유형 변경 → GPU 선택 후 재실행하세요.")

# ── 임베딩 모델 ───────────────────────────────────────────────────────────────
print("임베딩 모델 로드 중...")
_emb_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-large-instruct",
    encode_kwargs={"normalize_embeddings": True},
)
print("임베딩 모델 로드 완료")

# ── 지식베이스 → Chroma + BM25 ───────────────────────────────────────────────
_docs = []
for item in paper_rules:
    _docs.append(Document(
        page_content=json.dumps(item, ensure_ascii=False, indent=2),
        metadata={"type": "paper_rule"}
    ))
if shap_content:
    _docs.append(Document(page_content=shap_content, metadata={"type": "shap_analysis"}))

_splitter   = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=80)
_split_docs = _splitter.split_documents(_docs)

_database   = Chroma.from_documents(
    documents=_split_docs,
    embedding=_emb_model,
    collection_name="pragma_rag",
    persist_directory=CHROMA_DIR,
)
_vector_ret = _database.as_retriever(search_kwargs={"k": 5})
_bm25_ret   = BM25Retriever.from_documents(_split_docs)
_bm25_ret.k = 5

class _HybridRetriever(BaseRetriever):
    def _get_relevant_documents(
        self, query: str, *, run_manager: CallbackManagerForRetrieverRun = None
    ) -> _List[Document]:
        vec_docs  = _vector_ret.invoke(E5_PREFIX + query)
        bm25_docs = _bm25_ret.invoke(query)
        seen, combined = set(), []
        for doc in vec_docs + bm25_docs:
            key = doc.page_content[:80]
            if key not in seen:
                seen.add(key)
                combined.append(doc)
        return combined[:7]

    async def _aget_relevant_documents(self, query: str, **kwargs):
        return self._get_relevant_documents(query)

rag_retriever = _HybridRetriever()
print(f"RAG 구성 완료 (청크 수: {len(_split_docs)}, Chroma 경로: {CHROMA_DIR})")

# ── EXAONE LLM ────────────────────────────────────────────────────────────────
print("EXAONE LLM 로드 중... (최초 1회, 수 분 소요)")
_quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
_pipeline = HuggingFacePipeline.from_model_id(
    model_id="LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
    task="text-generation",
    pipeline_kwargs={"max_new_tokens": 512, "do_sample": False, "repetition_penalty": 1.03},
    model_kwargs={"quantization_config": _quant_cfg, "trust_remote_code": True},
)
exaone_llm = ChatHuggingFace(llm=_pipeline)
print("EXAONE LLM 로드 완료")

/tmp/ipykernel_1187/3406207286.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


임베딩 모델 로드 중...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/140k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


The repository LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct .
 You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
EXAONE LLM 로드 완료


## 7. Claude LLM 통합 (RAG + ML 예측 결과 기반 Q&A)

In [11]:
## 7-b. LOT 요약 + EXAONE Model RAG Q&A 함수

def build_lot_summary(lot_data: dict, pred_speed: float, X: pd.DataFrame, alerts: list) -> str:
    """LOT 공정 현황 요약 문자열 생성"""
    def safe_get(key, digits=4):
        v = lot_data.get(key)
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "N/A"
        return round(v, digits) if isinstance(v, float) else v

    alert_lines = []
    for a in alerts:
        if a["상태"] != "정상":
            alert_lines.append(
                f"  ⚠ {a['항목']}: {a['현재값']}{a['단위']} [{a['상태']}] "
                f"(LCL={a['LCL']}, SL={a['SL']}, UCL={a['UCL']})"
            )
    alert_str = "\n".join(alert_lines) if alert_lines else "  이탈 항목 없음"

    etch_cols = {
        "에칭 Cu 농도"  : safe_get(next((c for c in lot_data if "Etching" in c and "Cu" in c  and "분석치" in c), ""), 2),
        "에칭 HCl"      : safe_get(next((c for c in lot_data if "HCl" in c                   and "분석치" in c), ""), 3),
        "에칭 비중"      : safe_get(next((c for c in lot_data if "비중" in c                   and "분석치" in c), ""), 4),
        "에칭 온도"      : safe_get(next((c for c in lot_data if "온도" in c                   and "분석치" in c), ""), 2),
        "에칭 첨가제"    : safe_get(next((c for c in lot_data if "첨가제" in c                 and "분석치" in c), ""), 3),
        "에칭량"         : safe_get(next((c for c in lot_data if "Etching량" in c             and "분석치" in c), ""), 2),
    }
    etch_str = "\n".join(f"  {k}: {v}" for k, v in etch_cols.items())

    return f"""LOT 번호     : {lot_data.get('LOT')}
제품 코드    : {safe_get('통합코드')}
거래처       : {safe_get('거래처')}
공법 구분    : {safe_get('공법구분')}
LAYER        : {safe_get('LAYER')}
DRY FILM     : {safe_get('DRY FILM 정보')}

[에칭 공정 실측치]
{etch_str}
  Cu 표면두께 평균: {safe_get('Cu 표면두께 AVG_VAL')}

[ML 예측 결과]
  예측 에칭 속도 : {pred_speed:.4f} m/min
  실제 에칭 속도 : {safe_get('부식 Speed')} m/min
  검사 결과      : {safe_get('Result Ng2')}

[OPLS 공정 기준 대비 상태]
{alert_str}
"""


MODEL_RAG_TEMPLATE = """다음 문맥을 참고하여 질문에 답변해 주세요.

문맥에는 논문 기반 공정 rule, SHAP 기반 모델 해석 정보,
그리고 현재 LOT의 ML 예측 결과가 포함됩니다.

[현재 공정 실측 및 ML 예측 현황]
{lot_context}

[지식베이스 컨텍스트]
{knowledge_context}

답변 시 아래 내용을 중심으로 정리해 주세요.
- 현재 LOT 상태와 연계한 핵심 답변
- 관련 공정 변수
- 모델/문헌 기반 근거
- 조치 방향

질문:
{question}

답변:
"""


def ask_pragma(
    question  : str,
    lot_data  : dict,
    pred_speed: float,
    X         : pd.DataFrame,
    alerts    : list,
    verbose   : bool = True,
) -> str:
    """EXAONE Model RAG로 LOT 기반 공정 Q&A 수행"""
    lot_summary = build_lot_summary(lot_data, pred_speed, X, alerts)

    docs = rag_retriever.invoke(question)
    knowledge_context = "\n\n".join(
        f"[type={d.metadata.get('type','')}]\n{d.page_content}"
        for d in docs
    )

    prompt_text = MODEL_RAG_TEMPLATE.format(
        lot_context       = lot_summary,
        knowledge_context = knowledge_context,
        question          = question,
    )

    if verbose:
        print(f"[질문]: {question}")
        print("EXAONE 호출 중...")

    answer = exaone_llm.invoke(prompt_text).content

    if verbose:
        print("\n[답변]:")
        print(answer)

    return answer


print("build_lot_summary() + ask_pragma() 정의 완료")

build_lot_summary() + ask_pragma() 정의 완료


## 8. 통합 실행 함수

In [12]:
def analyze_lot(lotno: str, question: str) -> str:
    """LOT 번호 + 질문 → EXAONE Model RAG 공정 분석 답변"""
    print(f"\n{'='*60}")
    print(f" LOT {lotno} 분석 시작")
    print(f"{'='*60}")

    # Step 1: MES에서 공정 데이터 조회
    lot_data = get_lot_data(lotno)
    if lot_data is None:
        return f"LOT '{lotno}' 를 찾을 수 없습니다."

    # Step 2: LightGBM으로 에칭 속도 예측
    pred_speed, X = predict_etch_speed(lot_data)
    print(f"▶ 예측 에칭 속도: {pred_speed:.4f} m/min  "
          f"(실제: {lot_data.get('부식 Speed', 'N/A')} m/min)")

    # Step 3: OPLS 기준 이탈 체크
    alerts   = check_opls_status(X)
    abnormal = [a for a in alerts if a["상태"] != "정상"]
    if abnormal:
        print(f"▶ OPLS 이탈 항목: {len(abnormal)}개")
        for a in abnormal:
            print(f"   - {a['항목']}: {a['현재값']}{a['단위']} [{a['상태']}]")
    else:
        print("▶ OPLS 이탈 항목: 없음")

    # Step 4: EXAONE Model RAG로 답변 생성
    print()
    answer = ask_pragma(question, lot_data, pred_speed, X, alerts, verbose=True)
    return answer


print("analyze_lot() 정의 완료")

analyze_lot() 정의 완료


## 9. Streamlit 앱 코드 생성

In [77]:
from pathlib import Path

APP_CODE = r'''
import json
import pickle
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import streamlit as st


BASE_DIR = Path(__file__).parent
DATA_DIR = BASE_DIR / "data"
NOTEBOOK_DIR = BASE_DIR / "notebooks"
MODEL_DIR = BASE_DIR / "models"

MODEL_CANDIDATES = [
    NOTEBOOK_DIR / "best_LightGBM_mass_speed_regressor.pkl",
    MODEL_DIR / "best_LightGBM_mass_speed_regressor.pkl",
    BASE_DIR / "best_LightGBM_mass_speed_regressor.pkl",
]

SHAP_CANDIDATES = [
    NOTEBOOK_DIR / "shap_analysis_for_rag.md",
    DATA_DIR / "shap_analysis_for_rag.md",
    BASE_DIR / "shap_analysis_for_rag.md",
]

CSV_PATH = DATA_DIR / "Train_0319.csv"
PAPER_JSON_PATH = DATA_DIR / "rag_data_all.json"
OPLS_JSON_PATH = DATA_DIR / "opls_process_knowledge.json"


st.set_page_config(page_title="PRAGma", page_icon="⚙️", layout="wide")

st.markdown("""
<style>
.block-container {
    max-width: 1450px;
    padding-top: 1.2rem;
    padding-bottom: 2rem;
}

h1 {
    font-size: 2.5rem !important;
    font-weight: 800 !important;
}

.kpi-card {
    padding: 18px 20px;
    border-radius: 18px;
    color: white;
    min-height: 145px;
    box-shadow: 0 8px 20px rgba(15, 23, 42, 0.12);
}

.kpi-green {
    background: linear-gradient(135deg, #28544d 0%, #1e7565 100%);
}

.kpi-orange {
    background: linear-gradient(135deg, #654321 0%, #b26a12 100%);
}

.kpi-red {
    background: linear-gradient(135deg, #5f2d2d 0%, #b83434 100%);
}

.kpi-title {
    font-size: 15px;
    font-weight: 700;
    opacity: 0.85;
    margin-bottom: 14px;
}

.kpi-value {
    font-size: 34px;
    font-weight: 800;
    margin-bottom: 12px;
}

.kpi-status {
    display: inline-block;
    background: rgba(255,255,255,0.22);
    padding: 5px 14px;
    border-radius: 999px;
    font-size: 13px;
    font-weight: 700;
}

.left-menu {
    position: sticky;
    top: 1rem;
    background: #f8fafc;
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    padding: 16px;
}

.menu-title {
    font-weight: 800;
    margin-bottom: 12px;
}

.menu-item {
    padding: 8px 0;
    color: #334155;
    font-size: 14px;
}

.alert-red {
    background:#fff5f5;
    border-left:6px solid #ef4444;
    padding:16px;
    border-radius:12px;
    margin-bottom:14px;
}

.alert-yellow {
    background:#fffaf0;
    border-left:6px solid #f59e0b;
    padding:16px;
    border-radius:12px;
    margin-bottom:14px;
}

.result-box {
    background:#f8fafc;
    border:1px solid #e5e7eb;
    padding:16px;
    border-radius:12px;
    margin-bottom:14px;
}
</style>
""", unsafe_allow_html=True)


def find_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return None


@st.cache_resource
def load_resources():
    model_path = find_path(MODEL_CANDIDATES)
    if model_path is None:
        raise FileNotFoundError("best_LightGBM_mass_speed_regressor.pkl 파일을 찾을 수 없습니다.")

    with open(model_path, "rb") as f:
        model = pickle.load(f)

    df = pd.read_csv(CSV_PATH, encoding="cp949")

    paper_rules = []
    if PAPER_JSON_PATH.exists():
        with open(PAPER_JSON_PATH, "r", encoding="utf-8") as f:
            paper_rules = json.load(f)

    opls_rules = []
    if OPLS_JSON_PATH.exists():
        with open(OPLS_JSON_PATH, "r", encoding="utf-8") as f:
            opls_rules = json.load(f)

    shap_path = find_path(SHAP_CANDIDATES)
    shap_text = shap_path.read_text(encoding="utf-8") if shap_path else ""

    return model, df, paper_rules, opls_rules, shap_text, model_path


model, df_mes, paper_rules, opls_rules, shap_text, model_path = load_resources()

try:
    MODEL_FEATURES = model.feature_name_
except Exception:
    MODEL_FEATURES = model.booster_.feature_name()


CONTINUOUS_MAP = {
    "Cu 표면두께 Max_Val": "cu_thick_max",
    "Cu 표면두께 AVG_VAL": "cu_thick_avg",
    "Cu 표면두께 Min_Val": "cu_thick_min",
    "Cu 표면두께 Std_Val": "cu_thick_std",
    "Cu 표면두께 Median_Val": "cu_thick_median",
}

SUFFIX_MAP = {
    "Etch factor": "etch_factor",
    "Etching(염화동) - Cu": "meas_etch_cu",
    "Etching(염화동) - HCl": "meas_etch_hcl",
    "Etching(염화동) - 비중": "meas_etch_sg",
    "Etching(염화동) - 온도": "meas_etch_temp",
    "Etching-첨가제(HB-120EF)": "meas_etch_additive",
    "Etching량": "meas_etch_amount",
    "Soft Etch - Cu": "meas_softetch_cu",
    "Soft Etch - H2SO4": "meas_softetch_h2so4",
    "Soft Etch - SPS": "meas_softetch_sps",
    "박리액 - 농도": "meas_strip_conc",
    "수세수 - pH": "meas_rinse_ph",
    "현상액 - pH": "meas_dev_ph",
    "현상액 - 농도": "meas_dev_conc",
}

for col in df_mes.columns:
    if "분석치" in col or "分" in col:
        suffix = col.split("_", 1)[-1] if "_" in col else col
        if suffix in SUFFIX_MAP:
            CONTINUOUS_MAP[col] = SUFFIX_MAP[suffix]


CATEGORICAL_COLS = {
    "재작업사유": {
        "prefix": "rework_history",
        "values": ["Unknown", "기타", "기판 겹침", "두께 미달", "딤플", "설비 에러"],
        "sep": "_",
    },
    "노광 설비정보": {
        "prefix": "expo_eq_id",
        "values": [f"EXP-{i:03d}" for i in range(1, 8)],
        "sep": "_",
    },
    "DES 설비정보": {
        "prefix": "des_eq_id",
        "values": [f"DES-{i:03d}" for i in range(1, 7)],
        "sep": "_",
    },
    "정면 설비정보": {
        "prefix": "brush_eq_id",
        "values": [f"PRE-{i:03d}" for i in range(1, 8)],
        "sep": "_",
    },
}


OPLS_BOUNDS = {
    "meas_etch_temp": {"name": "에칭 온도", "lcl": 44.5, "sl": 48.0, "ucl": 53.0, "unit": "°C"},
    "meas_etch_sg": {"name": "에칭 비중", "lcl": 1.32, "sl": 1.37, "ucl": 1.42, "unit": ""},
    "meas_etch_cu": {"name": "에칭 Cu 농도", "lcl": 125.0, "sl": 155.0, "ucl": 185.0, "unit": "g/L"},
    "meas_etch_hcl": {"name": "에칭 HCl", "lcl": 0.3, "sl": 0.5, "ucl": 0.7, "unit": "N"},
    "meas_etch_additive": {"name": "에칭 첨가제", "lcl": 2.6, "sl": 3.0, "ucl": 3.4, "unit": "g/L"},
}


def get_lot_data(lot_no):
    rows = df_mes[df_mes["LOT"] == lot_no]
    if rows.empty:
        return None
    return rows.iloc[0].to_dict()


def prepare_features(lot_data):
    row = {}

    for csv_col, feat in CONTINUOUS_MAP.items():
        val = lot_data.get(csv_col, np.nan)
        row[feat] = float(val) if val is not None and str(val) not in ["", "nan", "None"] else np.nan

    for csv_col, cfg in CATEGORICAL_COLS.items():
        raw = str(lot_data.get(csv_col, "")).strip()
        norm = raw.replace(" ", "_") if cfg["prefix"] == "rework_history" else raw

        for value in cfg["values"]:
            norm_value = value.replace(" ", "_") if cfg["prefix"] == "rework_history" else value
            key = f"{cfg['prefix']}{cfg['sep']}{norm_value}"
            row[key] = 1 if norm == norm_value else 0

    for feat in MODEL_FEATURES:
        if feat not in row:
            row[feat] = 0

    return pd.DataFrame([row])[MODEL_FEATURES]


def predict_speed(lot_data):
    X = prepare_features(lot_data)
    pred = float(model.predict(X)[0])
    return pred, X


def check_opls(X):
    result = []

    for feat, bound in OPLS_BOUNDS.items():
        if feat not in X.columns:
            continue

        value = X[feat].iloc[0]
        if pd.isna(value):
            continue

        status = "정상"
        level = "ok"

        if value > bound["ucl"]:
            status = "UCL 초과"
            level = "high"
        elif value < bound["lcl"]:
            status = "LCL 미달"
            level = "high"
        elif value > bound["sl"] * 1.02:
            status = "SL 상향 근접"
            level = "mid"
        elif value < bound["sl"] * 0.98:
            status = "SL 하향 근접"
            level = "mid"

        result.append({
            "피처": feat,
            "항목": bound["name"],
            "현재값": round(float(value), 4),
            "LCL": bound["lcl"],
            "SL": bound["sl"],
            "UCL": bound["ucl"],
            "단위": bound["unit"],
            "상태": status,
            "위험도": level,
        })

    return result


def process_summary(alerts):
    high = [a for a in alerts if a["위험도"] == "high"]
    mid = [a for a in alerts if a["위험도"] == "mid"]

    if high:
        return "위험", f"{high[0]['항목']} {high[0]['상태']}", "즉시 점검이 필요합니다."
    if mid:
        return "주의", f"{mid[0]['항목']} {mid[0]['상태']}", "추세 확인 및 사전 점검이 필요합니다."
    return "정상", "OPLS 이탈 없음", "주요 관리 기준 내에 있습니다."


def recommend_speed(pred, actual):
    if not isinstance(actual, (int, float, np.integer, np.floating)):
        return "확인 불가", None, "실제 속도 데이터가 없어 예측값만 참고합니다."

    diff = pred - actual

    if abs(diff) < 0.03:
        return "유지", diff, "현재 속도와 모델 권장 속도 차이가 작아 유지 권장입니다."
    if diff > 0:
        return "상향", diff, f"현재 대비 약 {diff:.3f} m/min 상향 검토 가능합니다."
    return "하향", diff, f"현재 대비 약 {abs(diff):.3f} m/min 하향 검토가 필요합니다."


def search_knowledge(question, top_k=4):
    q = question.lower()
    docs = []

    for item in paper_rules:
        docs.append(("논문 Rule", json.dumps(item, ensure_ascii=False)))

    for item in opls_rules:
        docs.append(("OPLS Rule", json.dumps(item, ensure_ascii=False)))
        for chunk in item.get("rag_chunks", []):
            docs.append(("OPLS Chunk", chunk.get("content", "")))

    if shap_text:
        docs.append(("SHAP", shap_text[:3000]))

    keys = ["온도", "비중", "cu", "구리", "hcl", "첨가제", "선폭", "과에칭", "미에칭", "속도", "etch", "factor", "ph", "박리"]
    scored = []

    for doc_type, text in docs:
        low = text.lower()
        score = 0

        for key in keys:
            if key in q and key in low:
                score += 2

        for word in q.replace(",", " ").replace(".", " ").split():
            if len(word) >= 2 and word in low:
                score += 1

        if score > 0:
            scored.append((score, doc_type, text))

    scored.sort(reverse=True, key=lambda x: x[0])
    return scored[:top_k]


def chatbot_answer(question):
    docs = search_knowledge(question)

    lines = []
    lines.append("### 조치 방향")
    lines.append("- 센서값과 실측값을 먼저 교차 확인합니다.")
    lines.append("- 질문과 관련된 공정 변수의 OPLS 이탈 또는 근접 여부를 우선 점검합니다.")
    lines.append("- 약품 보충, 배액 라인, 칠러, 노즐, 펌프 에어록 여부를 순차적으로 확인합니다.")
    lines.append("")
    lines.append("### 관련 근거")

    if not docs:
        lines.append("- 관련 문서를 찾지 못했습니다.")
    else:
        for idx, (_, doc_type, text) in enumerate(docs[:3], start=1):
            text = text.replace("\n", " ")
            if len(text) > 350:
                text = text[:350] + "..."
            lines.append(f"{idx}. **{doc_type}**: {text}")

    return "\n".join(lines)


def rag_answer(question, alerts=None, pred=None):
    return chatbot_answer(question)


def trouble_actions(trouble):
    table = {
        "과에칭": ["에칭 온도/비중 UCL 초과 여부 확인", "Cu/HCl 농도 확인", "컨베이어 속도 하향 여부 확인", "선폭 OFFSET 마이너스 변동 확인"],
        "미에칭": ["Etch factor 저하 확인", "Cu 농도 과다 및 배액 불량 확인", "첨가제 농도 확인", "노즐 막힘 및 스프레이 압력 확인"],
        "선폭 불균일": ["Cu 표면두께 산포 확인", "Soft Etch SPS/H2SO4 확인", "롤러 마모 및 이송 속도 편차 확인"],
        "Cu 농도 과다": ["Auto Drain 확인", "신액 보충 상태 확인", "배액 라인 막힘 확인", "Etch factor 저하 동시 확인"],
        "Etch factor 저하": ["첨가제 실제 토출량 확인", "펌프 에어록 제거", "상하부 스프레이 압력 비교", "노즐 세척"],
        "현상액 pH 이탈": ["현상액 농도/pH 트렌드 확인", "K2CO3 보충 라인 확인", "보충 탱크 수위 확인"],
        "박리 잔사": ["박리액 농도 확인", "순환 펌프 필터 차압 확인", "스퀴지 롤러 상태 확인", "수세수 pH 확인"],
    }

    return table.get(trouble, [])


@st.cache_data
def make_monitoring_data():
    n = 30
    x = np.arange(n)

    temp = 48.0 + 0.10 * np.sin(x / 2) + np.linspace(-0.08, 0.15, n)
    sg = 1.370 + 0.004 * np.sin(x / 3) + np.linspace(0.000, 0.012, n)
    cu = 176 + 0.8 * x + 2.2 * np.sin(x / 4)
    additive = 2.95 + 0.035 * np.sin(x / 3) - np.linspace(0, 0.04, n)

    return pd.DataFrame({
        "시간": [f"T-{(n-i)*10}s" for i in range(n)],
        "에칭 온도": temp,
        "에칭 비중": sg,
        "에칭 Cu 농도": cu,
        "에칭 첨가제": additive,
    })


def make_deviation_chart(df, col, sl):
    return pd.DataFrame({
        "SL 대비 편차(%)": ((df[col] - sl) / sl) * 100
    })


def render_kpi_card(title, value, unit, status, color):
    st.markdown(
        f"""
        <div class="kpi-card {color}">
            <div class="kpi-title">{title}</div>
            <div class="kpi-value">{value}<span style="font-size:18px;"> {unit}</span></div>
            <div class="kpi-status">{status}</div>
        </div>
        """,
        unsafe_allow_html=True
    )


def render_dashboard():
    left_menu, main = st.columns([0.18, 0.82])

    with left_menu:
        st.markdown("""
        <div class="left-menu">
            <div class="menu-title">공정 대시보드</div>
            <div class="menu-item">〽 실시간 모니터링</div>
            <div class="menu-item">🔍 LOT 분석 / 최적 속도</div>
        </div>
        """, unsafe_allow_html=True)

    with main:
        st.subheader("〽 실시간 모니터링")

        df = make_monitoring_data()

        k1, k2, k3, k4 = st.columns(4)
        with k1:
            render_kpi_card("에칭 온도", f"{df['에칭 온도'].iloc[-1]:.2f}", "°C", "정상", "kpi-green")
        with k2:
            render_kpi_card("에칭 비중", f"{df['에칭 비중'].iloc[-1]:.3f}", "", "정상", "kpi-green")
        with k3:
            render_kpi_card("에칭 Cu 농도", f"{df['에칭 Cu 농도'].iloc[-1]:.1f}", "g/L", "UCL 초과", "kpi-red")
        with k4:
            render_kpi_card("에칭 첨가제", f"{df['에칭 첨가제'].iloc[-1]:.2f}", "g/L", "주의", "kpi-orange")

        st.write("")

        trend_col, alarm_col = st.columns([1.15, 1])

        with trend_col:
            st.markdown("#### 공정 트렌드")

            c1, c2 = st.columns(2)

            with c1:
                st.caption("에칭 온도 - SL 대비 편차")
                st.line_chart(make_deviation_chart(df, "에칭 온도", 48.0), height=180)

                st.caption("에칭 Cu 농도 - SL 대비 편차")
                st.line_chart(make_deviation_chart(df, "에칭 Cu 농도", 155.0), height=180)

            with c2:
                st.caption("에칭 비중 - SL 대비 편차")
                st.line_chart(make_deviation_chart(df, "에칭 비중", 1.37), height=180)

                st.caption("에칭 첨가제 - SL 대비 편차")
                st.line_chart(make_deviation_chart(df, "에칭 첨가제", 3.0), height=180)

        with alarm_col:
            st.markdown("#### 🔔 공정 알람")

            a1, a2, a3 = st.columns(3)
            a1.metric("위험", "1")
            a2.metric("경고", "2")
            a3.metric("해제", "5")

            st.markdown("""
            <div class="alert-red">
            <b>에칭 Cu 농도 UCL 초과 — DES-003</b><br><br>
            현재값 190.0 g/L | UCL 185.0 초과<br><br>
            권장 조치:
            <ul>
                <li>Auto Drain 확인</li>
                <li>신액 보충 진행</li>
                <li>Etch factor 동시 점검</li>
            </ul>
            </div>
            """, unsafe_allow_html=True)

            st.markdown("""
            <div class="alert-yellow">
            <b>에칭 첨가제 SL 하향 근접</b><br><br>
            첨가제 농도 저하 추세 확인<br><br>
            권장 조치:
            <ul>
                <li>첨가제 토출량 확인</li>
                <li>펌프 에어록 제거</li>
                <li>노즐 상태 점검</li>
            </ul>
            </div>
            """, unsafe_allow_html=True)

        st.divider()

        st.subheader("🔍 LOT 분석 / 최적 속도 도출")

        lot_left, lot_right = st.columns([1, 1.6])

        with lot_left:
            lot_no = st.text_input("LOT 번호", value="A20000")
            question = st.text_area(
                "분석 질문",
                value="이 LOT의 에칭 공정 상태와 개선 방향을 알려주세요.",
                height=120
            )

            lot_preview = get_lot_data(lot_no)

            if lot_preview is not None:
                with st.expander("LOT 기본 정보", expanded=True):
                    info_col1, info_col2 = st.columns(2)

                    with info_col1:
                        st.text_input("제품군", value=str(lot_preview.get("제품군", "N/A")), disabled=True)
                        st.text_input("거래처", value=str(lot_preview.get("거래처", "N/A")), disabled=True)
                        st.text_input("LAYER", value=str(lot_preview.get("LAYER", "N/A")), disabled=True)

                    with info_col2:
                        st.text_input("공법구분", value=str(lot_preview.get("공법구분", "N/A")), disabled=True)
                        st.text_input("도금구분", value=str(lot_preview.get("도금구분", "N/A")), disabled=True)
                        st.text_input("DRY FILM 정보", value=str(lot_preview.get("DRY FILM 정보", "N/A")), disabled=True)

                with st.expander("설비 정보", expanded=False):
                    st.text_input("노광 설비정보", value=str(lot_preview.get("노광 설비정보", "N/A")), disabled=True)
                    st.text_input("DES 설비정보", value=str(lot_preview.get("DES 설비정보", "N/A")), disabled=True)
                    st.text_input("정면 설비정보", value=str(lot_preview.get("정면 설비정보", "N/A")), disabled=True)

            run = st.button("분석 실행", type="primary", use_container_width=True)
            st.caption(f"사용 모델: {model_path.name}")

        with lot_right:
            if run:
                lot = get_lot_data(lot_no)

                if lot is None:
                    st.error(f"{lot_no} LOT를 찾을 수 없습니다.")
                else:
                    pred, X = predict_speed(lot)
                    actual = lot.get("부식 Speed")
                    alerts = check_opls(X)
                    risk, issue, desc = process_summary(alerts)
                    direction, diff, message = recommend_speed(pred, actual)

                    c1, c2, c3, c4 = st.columns(4)
                    c1.metric("예측 DES 속도", f"{pred:.4f} m/min")
                    c2.metric("실제 속도", f"{actual} m/min")

                    if isinstance(actual, (int, float, np.integer, np.floating)) and actual != 0:
                        c3.metric("오차율", f"{abs(pred - actual) / actual * 100:.2f}%")
                    else:
                        c3.metric("오차율", "N/A")

                    c4.metric("조정 방향", direction)

                    st.markdown(
                        f"""
                        <div class="result-box">
                        <b>{issue}</b><br>{desc}<br><br>
                        <b>속도 조정 제안</b><br>
                        {message}
                        </div>
                        """,
                        unsafe_allow_html=True
                    )

                    st.dataframe(pd.DataFrame(alerts), use_container_width=True)

                    st.markdown("#### AI 공정 분석")
                    st.markdown(rag_answer(question, alerts, pred))
            else:
                st.info("LOT 번호를 입력하고 분석 실행을 눌러주세요.")


top_left, top_right = st.columns([2, 1])

with top_left:
    st.title("⚙️ PRAGma")

with top_right:
    st.write("")
    st.success(f"● 실시간  |  {datetime.now().strftime('%H:%M:%S')}")

st.divider()

tab1, tab2, tab3 = st.tabs([
    "🏭 공정 대시보드",
    "🔧 트러블 대응",
    "💬 공정 지식 챗봇",
])


with tab1:
    render_dashboard()


with tab2:
    st.subheader("트러블 대응")

    left, right = st.columns([1, 1.4])

    with left:
        trouble = st.radio(
            "트러블 유형 선택",
            [
                "과에칭",
                "미에칭",
                "선폭 불균일",
                "Cu 농도 과다",
                "Etch factor 저하",
                "현상액 pH 이탈",
                "박리 잔사",
            ]
        )

        st.markdown(f"#### AI 조치 제안 — {trouble}")

        for idx, action in enumerate(trouble_actions(trouble), start=1):
            st.markdown(f"**{idx}. {action}**")

    with right:
        q = st.text_area(
            "공정 지식 검색",
            value=f"{trouble} 발생 시 원인과 조치 방향을 알려줘.",
            height=120
        )

        if st.button("지식 검색", type="primary", use_container_width=True):
            st.markdown(chatbot_answer(q))


with tab3:
    st.subheader("공정 지식 챗봇")

    q = st.text_input(
        "질문을 입력하세요",
        value="Etching 온도가 높으면 어떤 문제가 생기나요?"
    )

    examples = [
        "Cu 농도가 UCL 초과하면 어떤 조치를 해야 하나요?",
        "Etch factor가 낮을 때 원인은 무엇인가요?",
        "선폭 OFFSET이 마이너스일 때 무엇을 확인해야 하나요?",
        "현상액 pH가 낮으면 어떤 문제가 생기나요?",
    ]

    cols = st.columns(4)
    selected = None

    for col, example in zip(cols, examples):
        if col.button(example):
            selected = example

    if selected:
        q = selected

    if st.button("답변 생성", type="primary"):
        st.markdown(chatbot_answer(q))
'''

Path("/content/PRAGma/pragma_streamlit.py").write_text(APP_CODE, encoding="utf-8")
print("완료: /content/PRAGma/pragma_streamlit.py 생성됨")

완료: /content/PRAGma/pragma_streamlit.py 생성됨


## 10. Streamlit 서버 실행

In [78]:
import subprocess
import time

subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)

proc = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/PRAGma/pragma_streamlit.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(5)

print("Streamlit 서버 실행 완료")

Streamlit 서버 실행 완료


# 11. Cloudflare Tunnel 실행

In [79]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

In [80]:
import subprocess
import re

cf = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for _ in range(80):
    line = cf.stdout.readline()
    print(line, end="")

    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)

    if match:
        print("\n앱 주소:", match.group(0))
        break

2026-06-07T12:04:48Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-07T12:04:48Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-07T12:04:53Z INF +--------------------------------------------------------------------------------------------+
2026-06-07T12:04:53Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-07T12:04:53Z INF |  https://europe-precipitation-novelty-web.trycloudflar

streamlit cloud 세션 끊겼을때

In [64]:
import subprocess
import time
import re

# 기존 프로세스 종료
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
subprocess.run(["pkill", "-f", "cloudflared"], capture_output=True)

# Streamlit 실행
streamlit_proc = subprocess.Popen(
    [
        "streamlit",
        "run",
        "/content/PRAGma/pragma_streamlit.py",
        "--server.port=8501",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(5)

# Cloudflare 터널 실행
cf = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

for _ in range(80):
    line = cf.stdout.readline()
    print(line, end="")

    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)

    if match:
        print("\n새 앱 주소:", match.group(0))
        break

2026-06-07T11:32:58Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-07T11:32:58Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-07T11:33:02Z INF +--------------------------------------------------------------------------------------------+
2026-06-07T11:33:02Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-07T11:33:02Z INF |  https://jay-unto-flyer-increased.trycloudflare.com   